# Reproducibility notebook — ICDM 2026

**Trajectory-Aware Node Contributions and the Limits of Static Controllability**

This notebook reproduces, in run-order, every number the paper reports from the
real-data and synthetic experiments. Run the cells top to bottom on a machine with
the project's `code/` and `data/` folders mounted (see Setup). A CUDA GPU is
recommended but not required.

### What this notebook produces
| Section | Output | Used in paper |
|---|---|---|
| 1. Setup | imports, determinism seeding | — |
| 2. Data + split | leakage-safe pre-2000 V-Dem split | Sec. application |
| 3. NAVAR training | frozen `navar_model` (val_loss ≈ 0.043) | Sec. application |
| 4. Five-domain run | Table 2 (placement of five panels) | Table 2 |
| 5. Phase diagram | `phase_diagram_results.csv`, `phase_diagram.png` | Table 1, Fig. 1 |
| 6. Deep-domain ensemble | 20-seed per-node leverage stability | Sec. application (worked example) |

### Reproducibility notes
- **Determinism.** Section 1 fixes all RNG seeds (Python, NumPy, PyTorch, CUDA/cuDNN).
  Single-model training (Section 3) is seeded once; the seed-ensemble (Section 6)
  varies the seed deliberately and seeds each member.
- **Two controllability baselines.** Table 2 reports the departure of the node
  ordering from average controllability under two baselines: *realized* (average of
  the trajectory Jacobians) and *pooled* (a single linearization at the mean state).
  The realized baseline is the stable, apples-to-apples comparison and the one the
  paper leads on. For the democracy panel the pooled comparison is weakly identified
  (its rank correlation is near zero and varies in sign across retrainings), so the
  paper reports the democracy realized departure as the headline value and treats the
  pooled value as a secondary, unstable comparison rather than a point estimate.
- **Frozen synthetic DGP.** The phase-diagram data-generating process is fixed
  (`BGAIN=2.0`, `DIAG=0.6`, `NOISE=0.5`) and reported at 20 seeds.


## 1. Setup — paths, modules, and reproducible seeding

In [1]:
# =============================================================================
# COLAB SETUP — clone the repo, put src/ on the path, set data/output locations.
# This is the ONLY cell that needs the repository name. No Drive mount and no
# manual uploads: inputs come from the clone, outputs are written back into it.
# =============================================================================
import os, sys, subprocess
from pathlib import Path

REPO = "Emergent_Contribution"
USER = "author1181"                # anonymous author handle
REPO_URL = f"https://github.com/{USER}/{REPO}.git"

# Clone only if we're not already inside the repo (idempotent: safe to re-run).
if Path(REPO).exists():
    ROOT = Path(REPO).resolve()
elif Path("src").exists() and Path("data").exists():
    ROOT = Path(".").resolve()     # already running from inside the repo
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    ROOT = Path(REPO).resolve()

# Canonical locations, all inside the clone.
CODE    = ROOT / "src"
DATA    = ROOT / "data"
RESULTS = ROOT / "results"
OUT_DIR = RESULTS                  # bundle artifacts the verifier reads
RESULTS.mkdir(parents=True, exist_ok=True)

# Put src/ on the import path so `from train_NAVAR import ...` resolves.
sys.path.insert(0, str(CODE))

# GPU check.
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Repo root:", ROOT)
print("Code:     ", CODE, "exists:", CODE.exists())
print("Data:     ", DATA, "exists:", DATA.exists())
print("Outputs:  ", OUT_DIR)
print("CUDA available:", torch.cuda.is_available(),
      "| device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Repo root: /content/Emergent_Contribution
Code:      /content/Emergent_Contribution/src exists: True
Data:      /content/Emergent_Contribution/data exists: True
Outputs:   /content/Emergent_Contribution/results
CUDA available: True | device: cuda
GPU: Tesla T4


In [ ]:
# =============================================================================
# Import project modules (fails loudly if code/ is not on the path)
# =============================================================================
# If this fails, the modules are not on sys.path (check the setup cell ran).
from leakage_safe_split import build_cutoff_split, assess_power, _assert_no_leakage
from asof_jacobian import (
    make_torch_jacobian_fn, evaluate_asof_c,
    spectral_radius, leading_eigenvalue, bifurcation_label, finite_time_sigma,
)
import numpy as np
import pandas as pd
print("Modules imported OK.")


Modules imported OK.


In [ ]:
# -----------------------------------------------------------------------------
# Global reproducibility: fix all RNG seeds and enable deterministic kernels.
# Call seed_everything(seed) immediately before any model-training step. Single-
# model training uses the canonical GLOBAL_SEED; the seed-ensemble (Section 6)
# overrides it per member.
# -----------------------------------------------------------------------------
import os, random
import numpy as np
import torch

GLOBAL_SEED = 42   # canonical seed for the single-model (Table 2) NAVAR

def seed_everything(seed: int = GLOBAL_SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

seed_everything(GLOBAL_SEED)
print(f"Seeded everything with GLOBAL_SEED={GLOBAL_SEED}")


Seeded everything with GLOBAL_SEED=42


## 2. Load the V-Dem panel and build the leakage-safe split

The 16 V-Dem component indicators for 89 countries over 1950–2024. A pre-registered temporal cutoff at year 2000 defines the training window; standardization statistics are frozen on the training rows to prevent leakage.

In [ ]:
# =============================================================================
# Load the V-Dem panel and define the analysis variables
# =============================================================================
LONG_PANEL = DATA / 'my_data_75_years.csv'
assert LONG_PANEL.exists(), f"Missing data file: {LONG_PANEL}"

df = pd.read_csv(LONG_PANEL)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} cols")

TIME_COL = "year"
UNIT_COL = "country_id"
COMPONENTS = [c for c in df.columns if c not in (TIME_COL, UNIT_COL)]
print(f"Time col: {TIME_COL} | Unit col: {UNIT_COL} | {len(COMPONENTS)} components")
print("Components:", COMPONENTS)

# Choose the temporal cutoff. Pre-registration: roughly the last third of the
# span available for testing. For 1950–2024, C = 2000.
CUTOFF_C       = 2000
INNER_VAL_BAND = 5
MAXLAGS        = 3     # must match what you train NAVAR with



Loaded 6,675 rows x 18 cols
Time col: year | Unit col: country_id | 16 components
Components: ['Freedom_of_expression', 'Freedom_of_association', 'Suffrage', 'Clean_elections', 'Elected_officials', 'Individual_liberty', 'Judicial_constraints', 'Legislative_constraints', 'Civil_participation', 'Direct_vote', 'Local_government', 'Regional_government', 'Deliberative', 'Equal_access', 'Equal_distribution', 'Equal_protection']


In [ ]:
# =============================================================================
# Build the leakage-safe split and freeze preprocessing (frozen mu/sigma)
# =============================================================================
split = build_cutoff_split(
    df, components=COMPONENTS, cutoff_C=CUTOFF_C,
    time_col=TIME_COL, unit_col=UNIT_COL, inner_val_band=INNER_VAL_BAND,
)
print("Leakage-safe split built. Provenance:")
for k, v in split.provenance().items():
    print(f"  {k}: {v}")

# The frozen training matrix for NAVAR (standardized with frozen mu/sigma).
# IMPORTANT: NAVAR trains on the TRAIN rows only (year <= C - v).
train_df = split.train_df.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)
Y_train_raw  = train_df[COMPONENTS].to_numpy(float)
Y_train_norm = split.standardize(Y_train_raw)        # frozen mu/sigma

# split_spec = per-country training lengths, SAME ORDER as stacked rows.
split_spec = (
    train_df.groupby(UNIT_COL, sort=False)[TIME_COL].count().tolist()
)
assert sum(split_spec) == len(Y_train_norm), "split_spec / row order mismatch"
print(f"\nNAVAR training set: {len(split_spec)} countries, "
      f"{len(Y_train_norm)} rows, years <= {CUTOFF_C - INNER_VAL_BAND}")



Leakage-safe split built. Provenance:
  cutoff_C: 2000
  inner_val_band: 5
  standardization_fit_on: train_rows_le_C_minus_v
  n_train_rows: 4094
  n_inner_val_rows: 445
  n_test_rows: 2136
  n_train_units: 89
  n_test_units: 89

NAVAR training set: 89 countries, 4094 rows, years <= 1995


## 3. Train the V-Dem NAVAR (frozen hyperparameters, fixed seed)

A single neural vector-autoregression on the frozen training matrix. Seeding makes this model — and therefore the Table 2 numbers derived from it — exactly reproducible. The model is frozen (gradients off) before any leverage analysis.

In [ ]:
# =============================================================================
# Train NAVAR on the frozen pre-cutoff training data, then freeze the model
# =============================================================================
# Hyperparameters are frozen for this panel (a 32-node model fits the V-Dem panel
# better than a smaller, more-regularized alternative; see paper, reproducibility).
from train_NAVAR import train_NAVAR

seed_everything(GLOBAL_SEED)   # <-- makes the trained model reproducible

HIDDEN_NODES  = 32
HIDDEN_LAYERS = 1
DROPOUT       = 0.0
EPOCHS        = 400
LEARNING_RATE = 1e-3
BATCH_SIZE    = 256
LAMBDA1       = 0.15
WEIGHT_DECAY  = 1e-4

_, _, val_loss, navar_model = train_NAVAR(
    data             = Y_train_norm,        # FROZEN pre-C standardized data
    maxlags          = MAXLAGS,
    hidden_nodes     = HIDDEN_NODES,
    hidden_layers    = HIDDEN_LAYERS,
    dropout          = DROPOUT,
    epochs           = EPOCHS,
    learning_rate    = LEARNING_RATE,
    batch_size       = BATCH_SIZE,
    lambda1          = LAMBDA1,
    val_proportion   = 0.10,                 # inner split for checkpointing
    weight_decay     = WEIGHT_DECAY,
    check_every      = max(1, EPOCHS // 10),
    normalize        = False,                # already standardized; do NOT re-normalize
    split_timeseries = split_spec,
    lstm             = False,
)
navar_model.to(DEVICE).eval()
for p in navar_model.parameters():
    p.requires_grad_(False)
print(f"NAVAR trained and FROZEN. val_loss = {val_loss:.5f}")

# IMPORTANT CONVENTION CHECK: confirm the model call returns (preds, contribs)
# and that preds has shape (B, N). Run one probe.
N = len(COMPONENTS)
probe = torch.zeros(1, N, MAXLAGS, device=DEVICE)
with torch.no_grad():
    out = navar_model(probe)
preds_probe = out[0]
print("Model output OK. preds shape:", tuple(preds_probe.shape), "(expect (1, %d))" % N)


iteration 40. Loss: 0.29265809059143066  Val loss: 0.04909231886267662
iteration 80. Loss: 0.15744395554065704  Val loss: 0.04479622840881348
iteration 120. Loss: 0.15191662311553955  Val loss: 0.043611910194158554
iteration 160. Loss: 0.1503782570362091  Val loss: 0.043310485780239105
iteration 200. Loss: 0.14974579215049744  Val loss: 0.04317012429237366
iteration 240. Loss: 0.14914129674434662  Val loss: 0.0434279702603817
iteration 280. Loss: 0.14877380430698395  Val loss: 0.042329370975494385
iteration 320. Loss: 0.14853648841381073  Val loss: 0.042702414095401764
iteration 360. Loss: 0.1484254002571106  Val loss: 0.04257962107658386
iteration 400. Loss: 0.14834584295749664  Val loss: 0.042956605553627014
NAVAR trained and FROZEN. val_loss = 0.04296
Model output OK. preds shape: (1, 16) (expect (1, 16))


## 4. Canonical five-domain run — produces Table 2

Fits the same neural VAR to five public panels (democracy, macro-finance, development, realized volatility, air quality) and measures how far each panel's node-leverage ordering departs from average controllability under the realized and pooled baselines. The democracy row uses the seeded `navar_model` from Section 3; the other four are fit deterministically within the cell. Writes `domain_placement.csv`.

In [ ]:
# =============================================================================
# CANONICAL RUN (fully fixed) — every domain + BOTH controllability baselines
# + dominant-driver persistence & sign-flip mechanism check.
# Democracy uses the seeded navar_model from Section 3; the other four panels
# Needs in /data: my_data_75_years.csv, gmd_macro_core.csv, wdi_reversal_panel.csv,
#   rv_dataset.csv, PRSA_Data_20130301-20170228/PRSA_Data_*.csv
# Needs live: navar_model (V-Dem 16-comp), split, DEVICE
# =============================================================================
import sys, glob, os, numpy as np, pandas as pd, torch, warnings; warnings.filterwarnings('ignore')
from torch.func import jacrev, vmap
from scipy.stats import spearmanr
# (src/ already on sys.path via the setup cell)
from emergent_contribution import emergent_contribution_from_jacobians, average_controllability_gramian_trace
from beijing_estimator import fit_navar
from asof_jacobian import make_torch_jacobian_fn

DATA = str(DATA)   # from the setup cell (the cloned repo's data/ folder)
MAXLAGS, HORIZON = 3, 8
AGG=set("WLD EUU OED HIC LIC LMC UMC MIC EAS ECS LCN MEA NAC SAS SSF ARB CEB EAR TEA TEC TLA TMN TSA TSS IBD IBT IDA IDB IDX PRE FCS HPC LDC OSS PSS SST INX LTE EMU AFE AFW".split())
VDEM=['Freedom_of_expression','Freedom_of_association','Suffrage','Clean_elections','Elected_officials',
 'Individual_liberty','Judicial_constraints','Legislative_constraints','Civil_participation','Direct_vote',
 'Local_government','Regional_government','Deliberative','Equal_access','Equal_distribution','Equal_protection']

def companion_fn(model, N):
    def pw(w): p,_=model(w.reshape(1,N,MAXLAGS)); return p.reshape(-1)
    jr=jacrev(pw)
    def c(w):
        w=torch.as_tensor(np.asarray(w),dtype=torch.float32)              # FIX: ensure tensor
        Jb=jr(w);top=torch.cat([Jb[:,:,MAXLAGS-1-l] for l in range(MAXLAGS)],1)
        eye=torch.eye(N,dtype=top.dtype);zero=torch.zeros(N,N,dtype=top.dtype)
        rows=[torch.cat([eye if cc==(r-1) else zero for cc in range(MAXLAGS)],1) for r in range(1,MAXLAGS)]
        return torch.cat([top]+rows,0).detach().numpy()                   # return numpy (match make_torch_jacobian_fn)
    return c

def run_lengths(series):
    s=np.sign(series); s=s[s!=0]
    if len(s)==0: return [0]
    out=[];cur=1
    for k in range(1,len(s)):
        if s[k]==s[k-1]:cur+=1
        else:out.append(cur);cur=1
    out.append(cur);return out

def score_domain(name, Xz_segs, jac_fn, N, x_mean=None):
    Xall=np.vstack(Xz_segs);runs=[];pos=0
    for s in Xz_segs: runs.append((pos,pos+len(s)));pos+=len(s)
    acc={k:[] for k in range(N)};Cs=np.zeros((N*MAXLAGS,)*2);Cn=0;allTop=[];rowb=[];rp=0
    for (a,b) in runs:
        seg=Xall[a:b];L=len(seg)
        if L<MAXLAGS+HORIZON: continue
        Js=np.array([jac_fn(seg[s-MAXLAGS:s].T) for s in range(MAXLAGS,L)])
        allTop.append(Js[:,:N,:N*MAXLAGS]);rowb.append((rp,rp+len(Js)));rp+=len(Js);Cs+=Js.sum(0);Cn+=len(Js)
        for p in range(MAXLAGS-1,L-HORIZON+1):
            st=p+1;jseq=[Js[s-MAXLAGS] for s in range(st,st+HORIZON-1)]
            ec=emergent_contribution_from_jacobians(jseq,n_components=N,horizon=HORIZON,strict=True)
            for k in range(N): acc[k].append(ec.weights[k])
    Ej=np.array([np.mean(acc[k]) for k in range(N)])
    Cm=Cs/Cn; ctrl_r=np.array([average_controllability_gramian_trace(Cm,j,HORIZON) for j in range(N)])
    gap_realized=1-spearmanr(Ej,ctrl_r).correlation
    if x_mean is not None:
        A_static=jac_fn(np.tile(x_mean[:,None],(1,MAXLAGS)))
        ctrl_p=np.array([average_controllability_gramian_trace(A_static,j,HORIZON) for j in range(N)])
        gap_pooled=1-spearmanr(Ej,ctrl_p).correlation
    else: gap_pooled=float('nan')
    full=np.vstack(allTop);prof=np.zeros(N)
    for lag in range(MAXLAGS): prof+=np.abs(full[:,:,lag*N:(lag+1)*N]).sum(axis=(0,1))
    dom=int(np.argmax(prof))
    E=np.zeros((len(full),N,N))
    for lag in range(MAXLAGS): E+=full[:,:,lag*N:(lag+1)*N]
    emag=np.array([np.mean(np.abs(E[:,i,dom])) for i in range(N)]);emag[dom]=-1;tgt=int(np.argmax(emag))
    signed=np.array([np.sum(E[t][:,dom])-E[t][dom,dom] for t in range(len(E))])
    cancel=abs(np.mean(signed))/(np.mean(np.abs(signed))+1e-12)
    rl=[]
    for (ra,rb) in rowb: rl.extend(run_lengths(E[ra:rb,tgt,dom]))
    rl=np.array(rl)
    return dict(name=name,N=N,gap_realized=gap_realized,gap_pooled=gap_pooled,dom=dom,
                cancel=cancel,med=float(np.median(rl)),frac=float(np.mean(rl>=HORIZON)))

def load_panel(file,vars,unit,time,level,strip_agg):
    df=pd.read_csv(f"{DATA}/{file}")
    if strip_agg: df=df[~df[unit].isin(AGG)]
    if unit: df=df.sort_values([unit,time])
    if level=='ALL_LOG':
        X=np.log(df[vars].clip(lower=1e-9));X=(X-X.mean())/X.std();return [X.dropna().values]
    if level:
        for v in level: df[v]=df.groupby(unit)[v].transform(lambda s:np.log(s.where(s>0)).diff()*100)
    segs=[]
    for _,g in df.groupby(unit,sort=False):
        g=g.sort_values(time);ok=g[vars].notna().all(axis=1).values;arr=g[vars].values;i=0
        while i<len(ok):
            if ok[i]:
                j=i
                while j<len(ok) and ok[j]:j+=1
                if j-i>=MAXLAGS+HORIZON+4: segs.append(arr[i:j])
                i=j
            else:i+=1
    X=np.vstack(segs);mu,sd=X.mean(0),X.std(0)
    return [(s-mu)/sd for s in segs]

R=[]
# ---- Democracy (V-Dem): uses the Section-3 navar_model ----
try:
    segs=[];df=pd.read_csv(f"{DATA}/my_data_75_years.csv").sort_values(['country_id','year'])
    for _,g in df.groupby('country_id',sort=False):
        g=g.sort_values('year');z=split.standardize(g[VDEM].to_numpy(float))
        if len(z)>=MAXLAGS+HORIZON: segs.append(z)
    jac=make_torch_jacobian_fn(navar_model,n_components=16,maxlags=MAXLAGS,lag_reduction="companion",device=DEVICE)
    R.append((score_domain('V-Dem',segs,lambda w:jac(w),16,x_mean=np.vstack(segs).mean(0)),'navar_model'))
    print("V-Dem done")
except Exception as e: print("SKIP V-Dem:",str(e)[:120])

# ---- macro / WDI / vol: fit_navar per domain ----
specs=[('GMD-macro','gmd_macro_core.csv',['rGDP','infl','cbrate','unemp','govdef','govdebt','CA_GDP','cons','inv'],'ISO3','year',['rGDP','cons','inv'],False),
       ('WDI','wdi_reversal_panel.csv',['gdp_growth','resource_rents','investment','trade_openness','fdi','inflation','unemployment','pop_growth'],'ISO3','year',[],True),
       ('realized-vol','rv_dataset.csv',['.SPX','.GDAXI','.FCHI','.FTSE','.OMXSPI','.N225','.KS11','.HSI'],None,None,'ALL_LOG',False)]
for nm,f,v,u,t,lv,sa in specs:
    try:
        if not os.path.exists(f"{DATA}/{f}"): print(f"SKIP {nm}: file missing"); continue
        segs=load_panel(f,v,u,t,lv,sa);N=len(v)
        runs=[];pos=0
        for s in segs: runs.append((pos,pos+len(s)));pos+=len(s)
        m,_,_=fit_navar(np.vstack(segs),runs,n_components=N,maxlags=MAXLAGS,epochs=400,seed=0)
        R.append((score_domain(nm,segs,companion_fn(m,N),N,x_mean=np.vstack(segs).mean(0)),'fit_navar'))
        print(f"{nm} done")
    except Exception as e: print(f"SKIP {nm}:",str(e)[:120])

# ---- Beijing: own prep + fit_navar ----
try:
    import beijing_prep
    bsegs=[]
    for p in sorted(glob.glob(f"{DATA}/PRSA_Data_20130301-20170228/PRSA_Data_*.csv")):
        _,Xz,rr=beijing_prep.load_site(p)
        for (a,b) in rr: bsegs.append(Xz[a:b])
    if bsegs:
        N=len(beijing_prep.NODES);runs=[];pos=0
        for s in bsegs: runs.append((pos,pos+len(s)));pos+=len(s)
        m,_,_=fit_navar(np.vstack(bsegs),runs,n_components=N,maxlags=MAXLAGS,epochs=400,seed=0)
        R.append((score_domain('Beijing',bsegs,companion_fn(m,N),N,x_mean=np.vstack(bsegs).mean(0)),'fit_navar'))
        print("Beijing done")
    else: print("SKIP Beijing: no site files found at",f"{DATA}/PRSA_Data_20130301-20170228/")
except Exception as e: print("SKIP Beijing:",str(e)[:120])

# ---- table ----
print(f"\n{'domain':>13s} {'estim':>11s} {'gap_realiz':>10s} {'gap_pooled':>10s} {'driver':>14s} {'cancel':>7s} {'med_run':>8s} {'frac>=8':>8s}")
for r,est in R:
    dn=(VDEM[r['dom']] if r['name']=='V-Dem' else f"idx{r['dom']}")[:14]
    print(f"{r['name']:>13s} {est:>11s} {r['gap_realized']:10.3f} {r['gap_pooled']:10.3f} {dn:>14s} {r['cancel']:7.2f} {r['med']:8.0f} {r['frac']:8.2f}")
print("\ncancel~1=sign-stable driver(no flip); low=sign-flipping. med_run/frac>=8=persistence.")
# (per-domain regime diagnostics: cancel = sign-stability of the dominant driver; med_run/frac>=8 = persistence)

# ---- EMIT canonical domain_placement.csv for the paper-number verifier ----
import pandas as pd
_KEYMAP = {'V-Dem':'democracy', 'GMD-macro':'macro', 'WDI':'development',
           'realized-vol':'realized_vol', 'Beijing':'air_quality'}
_rows = [{'domain': _KEYMAP.get(r['name'], r['name']),
          'n': r['N'],
          'realized': round(r['gap_realized'], 3),
          'pooled':   round(r['gap_pooled'], 3)}
         for r, est in R]
import os as _os
assert _rows, (
    'No domains were scored -- every panel was skipped. This almost always '
    'means the data files are not where DATA points. DATA is currently: '
    + str(DATA) + '  (expected the cloned repo data/ folder containing '
    'my_data_75_years.csv, gmd_macro_core.csv, rv_dataset.csv, '
    'wdi_reversal_panel.csv, and the Beijing site files).'
)
_dom_path = _os.path.join(str(OUT_DIR), 'domain_placement.csv')
pd.DataFrame(_rows)[['domain','n','realized','pooled']].to_csv(_dom_path, index=False)
print(f"\nwrote {_dom_path}:")
print(pd.DataFrame(_rows).to_string(index=False))

V-Dem done
GMD-macro done
WDI done
realized-vol done
Beijing done

       domain       estim gap_realiz gap_pooled         driver  cancel  med_run  frac>=8
        V-Dem navar_model      0.038      0.962 Freedom_of_exp    0.87       12     0.61
    GMD-macro   fit_navar      0.150      0.117           idx1    1.00       28     1.00
          WDI   fit_navar      0.000      0.238           idx6    0.31        2     0.16
 realized-vol   fit_navar      0.024      0.167           idx0    1.00     2612     1.00
      Beijing   fit_navar      0.009      0.045           idx6    0.96        6     0.49

cancel~1=sign-stable driver(no flip); low=sign-flipping. med_run/frac>=8=persistence.

wrote /content/Emergent_Contribution/results/domain_placement.csv:
      domain  n  realized  pooled
   democracy 16     0.038   0.962
       macro  9     0.150   0.117
 development  8     0.000   0.238
realized_vol  8     0.024   0.167
 air_quality 11     0.009   0.045


In [ ]:
# =============================================================================
# SEED-STABILITY CHECK across ALL fit_navar domains (macro, WDI, realized-vol,
# Beijing) at once. For each domain, re-fits under several seeds and reports the
# realized & pooled gap distribution, so Table 2 can report every fit_navar
# number honestly (a stable point, or a mean +/- s.d. if seed-sensitive).
#
# Run AFTER the canonical five-domain cell, while in scope:
#   DATA, MAXLAGS, HORIZON, fit_navar, companion_fn, score_domain, load_panel,
#   glob, np, os  (and beijing_prep importable)
#
# NOTE: Beijing re-fits on hourly multi-site data and is SLOW (~30 min/seed).
#       Set BEIJING_SEEDS small (e.g. 3) or skip if you only need the others.
# =============================================================================
import numpy as np, glob, os

N_SEEDS_FAST = 8     # macro / WDI / realized-vol
N_SEEDS_BEIJING = 3  # Beijing is expensive; fewer seeds

def summarize(name, segs, N, n_seeds):
    runs = []; pos = 0
    for s in segs: runs.append((pos, pos+len(s))); pos += len(s)
    x_mean = np.vstack(segs).mean(0)
    GR, GP = [], []
    for seed in range(n_seeds):
        m,_,_ = fit_navar(np.vstack(segs), runs, n_components=N,
                          maxlags=MAXLAGS, epochs=400, seed=seed)
        r = score_domain(name, segs, companion_fn(m, N), N, x_mean=x_mean)
        GR.append(r['gap_realized']); GP.append(r['gap_pooled'])
        print(f"    {name:>12s} seed {seed}: realized {r['gap_realized']:.3f} | pooled {r['gap_pooled']:.3f}")
    GR, GP = np.array(GR), np.array(GP)
    return dict(name=name, n_seeds=n_seeds,
                r_mean=GR.mean(), r_sd=GR.std(), r_min=GR.min(), r_max=GR.max(),
                p_mean=GP.mean(), p_sd=GP.std(), p_min=GP.min(), p_max=GP.max())

results = []

# ---- macro / WDI / realized-vol ----
specs=[('GMD-macro','gmd_macro_core.csv',['rGDP','infl','cbrate','unemp','govdef','govdebt','CA_GDP','cons','inv'],'ISO3','year',['rGDP','cons','inv'],False),
       ('WDI','wdi_reversal_panel.csv',['gdp_growth','resource_rents','investment','trade_openness','fdi','inflation','unemployment','pop_growth'],'ISO3','year',[],True),
       ('realized-vol','rv_dataset.csv',['.SPX','.GDAXI','.FCHI','.FTSE','.OMXSPI','.N225','.KS11','.HSI'],None,None,'ALL_LOG',False)]
for nm,f,v,u,t,lv,sa in specs:
    if not os.path.exists(f"{DATA}/{f}"):
        print(f"SKIP {nm}: file missing"); continue
    print(f"\n=== {nm} ===")
    segs = load_panel(f, v, u, t, lv, sa); N = len(v)
    results.append(summarize(nm, segs, N, N_SEEDS_FAST))

# ---- Beijing (slow) ----
try:
    import beijing_prep
    print(f"\n=== Beijing (slow: {N_SEEDS_BEIJING} seeds) ===")
    bsegs = []
    for p in sorted(glob.glob(f"{DATA}/PRSA_Data_20130301-20170228/PRSA_Data_*.csv")):
        _, Xz, rr = beijing_prep.load_site(p)
        for (a,b) in rr: bsegs.append(Xz[a:b])
    if bsegs:
        results.append(summarize('Beijing', bsegs, len(beijing_prep.NODES), N_SEEDS_BEIJING))
    else:
        print("SKIP Beijing: no site files found")
except Exception as e:
    print("SKIP Beijing:", str(e)[:120])

# ---- verdict table ----
print("\n" + "="*78)
print(f"{'domain':>13s} | {'realized mean±sd':>18s} {'[min,max]':>14s} | {'verdict':>20s}")
print("="*78)
for d in results:
    rng = f"[{d['r_min']:.3f},{d['r_max']:.3f}]"
    # heuristic: sd < 0.01 AND range width < 0.02 => stable point; else distribution
    stable = (d['r_sd'] < 0.01) and (d['r_max'] - d['r_min'] < 0.02)
    verdict = "STABLE point" if stable else "NOISY -> report mean±sd"
    print(f"{d['name']:>13s} | {d['r_mean']:.3f} ± {d['r_sd']:.3f}   {rng:>14s} | {verdict:>20s}")
print("="*78)
print("STABLE  -> report the single value in Table 2.")
print("NOISY   -> report mean±sd (like macro & democracy), with wide verifier tol.")
print("(development at the 0.00 floor is effectively stable: Spearman ~1, can't move down.)")


=== GMD-macro ===
       GMD-macro seed 0: realized 0.150 | pooled 0.117
       GMD-macro seed 1: realized 0.100 | pooled 0.100
       GMD-macro seed 2: realized 0.050 | pooled 0.050
       GMD-macro seed 3: realized 0.117 | pooled 0.133
       GMD-macro seed 4: realized 0.050 | pooled 0.067
       GMD-macro seed 5: realized 0.117 | pooled 0.150
       GMD-macro seed 6: realized 0.050 | pooled 0.067
       GMD-macro seed 7: realized 0.083 | pooled 0.100

=== WDI ===
             WDI seed 0: realized 0.000 | pooled 0.238
             WDI seed 1: realized 0.024 | pooled 0.071
             WDI seed 2: realized 0.000 | pooled 0.095
             WDI seed 3: realized 0.024 | pooled 0.214
             WDI seed 4: realized 0.024 | pooled 0.048
             WDI seed 5: realized 0.024 | pooled 0.024
             WDI seed 6: realized 0.024 | pooled 0.095
             WDI seed 7: realized 0.024 | pooled 0.333

=== realized-vol ===
    realized-vol seed 0: realized 0.024 | pooled 0.167
    realize

## 5. Phase diagram (synthetic, frozen DGP, 20 seeds)

Sweeps a two-regime nonlinear system across nonlinearity, regime structure, persistence, and perturbation amplitude, comparing emergent contribution and average controllability against ground-truth contribution over 20 seeds. Produces Table 1 and Figure 1. The data-generating process is frozen (`BGAIN=2.0`, `DIAG=0.6`, `NOISE=0.5`). Writes `phase_diagram_results.csv` and `phase_diagram.png`.

In [ ]:
# =============================================================================
# Phase-diagram grid at 20 seeds. Frozen DGP and metrics.
# Writes phase_diagram_results.csv (the artifact the paper's Table 1 reads) and
# phase_diagram.png (Figure 1).
# =============================================================================
import sys, numpy as np, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, pandas as pd
# (src/ already on sys.path via the setup cell)
from scipy.stats import spearmanr, wilcoxon
from emergent_contribution import emergent_contribution_from_jacobians, average_controllability_gramian_trace
np.seterr(all='ignore')

N, HORIZON, DRIVER = 7, 8, 0
BGAIN, DIAG, NOISE = 2.0, 0.6, 0.5            # FROZEN calibration — do not change

def make_regimes(seed, structure):
    r=np.random.default_rng(seed)
    A=r.normal(0,0.04,(N,N)); np.fill_diagonal(A,r.uniform(DIAG-0.05,DIAG+0.05,N)); B=r.normal(0,0.04,(N,N))
    A1,B1=A.copy(),B.copy(); A2,B2=A.copy(),B.copy()
    for t in (1,2,3): B1[t,DRIVER]=BGAIN
    if structure in ('smooth-drift','persistent-switch','sign-flip-switch'):
        for t in (1,2,3): B2[t,DRIVER]=(-BGAIN if 'sign-flip' in structure else BGAIN)
        if structure=='persistent-switch':
            for t in (1,2,3): B2[t,DRIVER]=0.0
            for t in (1,2,3): B2[t,4]=BGAIN
    for M in (A1,A2):
        sr=np.max(np.abs(np.linalg.eigvals(M)))
        if sr>0.85: M*=0.85/sr
    return (A1,B1),(A2,B2)
def step(reg,x,alpha): A,B=reg; return A@x+alpha*np.tanh(B@x)
def jac(reg,x,alpha): A,B=reg; pre=B@x; return A+alpha*(np.diag(1-np.tanh(pre)**2)@B)
def regime_at(t,L,regs,structure,which):
    if structure=='static': return regs[0]
    if structure=='smooth-drift':
        g=min(1.0,t/L); return (g*regs[1][0]+(1-g)*regs[0][0], g*regs[1][1]+(1-g)*regs[0][1])
    return regs[which[min(t,len(which)-1)]]
def simulate(regs,structure,alpha,persistence,T=400,seed=0):
    r=np.random.default_rng(seed+7); x=r.normal(0,1.0,N); xs=[x.copy()]; which=[]; cur=0
    p={'short':0.30,'medium':0.12,'long':0.02}[persistence]
    for t in range(T):
        if structure not in ('static','smooth-drift') and r.random()<p: cur=1-cur
        reg=regime_at(t,T,regs,structure,which+[cur]); x=step(reg,x,alpha)+r.normal(0,NOISE,N)
        xs.append(x.copy()); which.append(cur)
        if not np.all(np.isfinite(x)) or np.max(np.abs(x))>1e4: return None,None
    return np.array(xs),which
def true_contrib(xs,regs,structure,which,alpha,eps):
    acc=np.zeros(N)
    for t in range(len(xs)-HORIZON+1):
        x0=xs[t]
        for j in range(N):
            e=np.zeros(N);e[j]=eps;xp=x0+e;xb=x0.copy();en=np.sum((xp-xb)**2)
            for h in range(HORIZON-1):
                reg=regime_at(t+h,len(xs),regs,structure,which);xp=step(reg,xp,alpha);xb=step(reg,xb,alpha);en+=np.sum((xp-xb)**2)
            acc[j]+=en
    return acc/acc.sum()
def cell(structure,alpha,persistence,eps,seed):
    regs=make_regimes(seed,structure); xs,which=simulate(regs,structure,alpha,persistence,seed=seed)
    if xs is None: return None
    tr=true_contrib(xs,regs,structure,which,alpha,eps)
    Js=[jac(regime_at(t,len(xs),regs,structure,which),xs[t],alpha) for t in range(len(xs)-1)]
    acc={k:[] for k in range(N)}
    for t in range(len(Js)-HORIZON+2):
        jseq=[Js[t+h] for h in range(HORIZON-1)]
        ec=emergent_contribution_from_jacobians(jseq,n_components=N,horizon=HORIZON,strict=True)
        for k in range(N): acc[k].append(ec.weights[k])
    Ej=np.array([np.mean(acc[k]) for k in range(N)])
    Jmean=np.mean(Js,0)
    ACr=np.array([average_controllability_gramian_trace(Jmean,j,HORIZON) for j in range(N)])
    Abar=0.5*(regs[0][0]+regs[1][0]); Bbar=0.5*(regs[0][1]+regs[1][1])
    Jp=jac((Abar,Bbar),xs.mean(0),alpha)
    ACp=np.array([average_controllability_gramian_trace(Jp,j,HORIZON) for j in range(N)])
    return (spearmanr(Ej,tr).correlation,spearmanr(ACr,tr).correlation,
            spearmanr(ACp,tr).correlation,spearmanr(Ej,ACr).correlation)

NL={'low':0.3,'med':0.8,'high':1.5}
STRUCT=['static','smooth-drift','persistent-switch','sign-flip-switch']
PERS=['short','medium','long']
EPS={'small':0.02,'med':1.0,'large':6.0,'extreme':12.0}
SEEDS=range(20)                                 # <-- ONLY CHANGE: 8 -> 20
NO_SWITCH={'static','smooth-drift'}

def paired_ci(g,B=2000):
    g=np.asarray(g)
    if len(g)<2: return (np.nan,np.nan)
    bs=[np.mean(np.random.choice(g,len(g),replace=True)) for _ in range(B)]
    return (np.percentile(bs,2.5),np.percentile(bs,97.5))

rows=[]
for st in STRUCT:
  for nlk,al in NL.items():
    for pe in PERS:
      for epk,ep in EPS.items():
        rs=[cell(st,al,pe,ep,s) for s in SEEDS]; rs=[r for r in rs if r and np.all(np.isfinite(r))]
        if not rs: continue
        rs=np.array(rs); gr=rs[:,0]-rs[:,1]; gp=rs[:,0]-rs[:,2]
        cir=paired_ci(gr); cip=paired_ci(gp)
        try: wp=wilcoxon(rs[:,0],rs[:,2]).pvalue if len(rs)>=6 else np.nan
        except: wp=np.nan
        m=rs.mean(0)
        rows.append(dict(struct=st,nl=nlk,pers=pe,eps=epk,ej_t=m[0],acr_t=m[1],acp_t=m[2],ej_acr=m[3],
                         d_real=gr.mean(),d_pool=gp.mean(),d_real_lo=cir[0],d_real_hi=cir[1],
                         d_pool_lo=cip[0],d_pool_hi=cip[1],wilcoxon_p=wp,n=len(rs),
                         degenerate_control=(st in NO_SWITCH)))
        print(f"{st:>16s} {nlk:>4s} {pe:>6s} {epk:>7s} | Dpool {gp.mean():.3f} [{cip[0]:.3f},{cip[1]:.3f}] n={len(rs)}")
df=pd.DataFrame(rows)
import os as _os
df.to_csv(_os.path.join(str(OUT_DIR),"phase_diagram_results.csv"), index=False)
print("\nsaved phase_diagram_results.csv (20 seeds)")
fig,axes=plt.subplots(len(STRUCT),len(NL),figsize=(11,12),sharex=True,sharey=True)
epk_order=list(EPS); xpos=range(len(epk_order))
for i,st in enumerate(STRUCT):
  for j,nlk in enumerate(NL):
    ax=axes[i][j]
    for pe in PERS:
      sub=[df[(df.struct==st)&(df.nl==nlk)&(df.pers==pe)&(df.eps==ek)] for ek in epk_order]
      ejt=[(s.ej_t.values[0] if len(s) else np.nan) for s in sub]
      acrt=[(s.acr_t.values[0] if len(s) else np.nan) for s in sub]
      acpt=[(s.acp_t.values[0] if len(s) else np.nan) for s in sub]
      ax.plot(xpos,ejt,'-o',label=f'E_j ({pe})',alpha=0.8)
      ax.plot(xpos,acrt,'--x',label=f'AC_real ({pe})',alpha=0.45)
      ax.plot(xpos,acpt,':s',label=f'AC_pool ({pe})',alpha=0.35,markersize=3)
    ax.set_title(f'{st}\n{nlk} nonlin',fontsize=8); ax.set_ylim(0.3,1.05)
    if i==len(STRUCT)-1: ax.set_xticks(xpos); ax.set_xticklabels(epk_order,fontsize=7,rotation=45)
    if i==0 and j==len(NL)-1: ax.legend(fontsize=4,ncol=2)
fig.suptitle('Phase diagram: fidelity to true contribution — E_j (solid) vs AC_realized (dashed) vs AC_pooled (dotted)',fontsize=10)
fig.supxlabel('perturbation amplitude'); fig.supylabel('Spearman fidelity to ground truth')
plt.tight_layout(); plt.savefig(_os.path.join(str(OUT_DIR),"phase_diagram.png"), dpi=150, bbox_inches='tight')
print("saved phase_diagram.png")

          static  low  short   small | Dpool 0.007 [-0.005,0.021] n=20
          static  low  short     med | Dpool 0.007 [-0.005,0.021] n=20
          static  low  short   large | Dpool 0.007 [-0.009,0.023] n=20
          static  low  short extreme | Dpool 0.011 [-0.004,0.027] n=20
          static  low medium   small | Dpool 0.007 [-0.005,0.021] n=20
          static  low medium     med | Dpool 0.007 [-0.004,0.021] n=20
          static  low medium   large | Dpool 0.007 [-0.007,0.023] n=20
          static  low medium extreme | Dpool 0.011 [-0.005,0.025] n=20
          static  low   long   small | Dpool 0.007 [-0.005,0.021] n=20
          static  low   long     med | Dpool 0.007 [-0.005,0.021] n=20
          static  low   long   large | Dpool 0.007 [-0.007,0.025] n=20
          static  low   long extreme | Dpool 0.011 [-0.004,0.027] n=20
          static  med  short   small | Dpool 0.050 [0.007,0.098] n=20
          static  med  short     med | Dpool 0.050 [0.007,0.098] n=20
        

In [ ]:
# =============================================================================
# PAPER FIGURE — collapse the phase grid over persistence into the clean
# 3-line-per-panel figure (E_j vs AC-realized vs AC-pooled). Re-plots the
# EXISTING phase_diagram_results.csv; recomputes nothing. Overwrites
# phase_diagram.png so the regenerated figure matches the published one.
#
# Each line is the MEAN over the three persistence levels (short/medium/long)
# at each (structure, nonlinearity, amplitude) cell — the secondary persistence
# axis is summarized away to foreground the three measures, matching the paper.
# Layout: rows = regime structure, columns = nonlinearity (as in the paper).
# =============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

CSV = os.path.join(str(OUT_DIR), "phase_diagram_results.csv")
df = pd.read_csv(CSV)

# canonical orderings (rows/cols/x-axis) to match the paper
STRUCT = ['static', 'smooth-drift', 'persistent-switch', 'sign-flip-switch']
ROW_LABEL = {'static':'Static', 'smooth-drift':'Smooth drift',
             'persistent-switch':'Persistent reroute', 'sign-flip-switch':'Sign reversal'}
NL  = ['low', 'med', 'high']
NL_LABEL = {'low':'low nonlinearity', 'med':'med nonlinearity', 'high':'high nonlinearity'}
EPS = ['small', 'med', 'large', 'extreme']
xpos = range(len(EPS))

# mean over persistence: one value per (struct, nl, eps) for each measure
agg = (df.groupby(['struct', 'nl', 'eps'])[['ej_t', 'acr_t', 'acp_t']]
         .mean().reset_index())

# paper styling: E_j solid green w/ circles; AC-realized dashed blue w/ squares;
# AC-pooled dotted red w/ triangles.
SERIES = [('ej_t',  r'$E_j$',          '-',  'o', '#1b5e20'),
          ('acr_t', 'AC (realized)',   '--', 's', '#1f77b4'),
          ('acp_t', 'AC (pooled)',     ':',  '^', '#d62728')]

fig, axes = plt.subplots(len(STRUCT), len(NL), figsize=(8.5, 9.5),
                         sharex=True, sharey=True)
for i, st in enumerate(STRUCT):
    for j, nlk in enumerate(NL):
        ax = axes[i][j]
        cell = agg[(agg.struct == st) & (agg.nl == nlk)].set_index('eps')
        for col, lab, ls, mk, color in SERIES:
            y = [cell.loc[e, col] if e in cell.index else np.nan for e in EPS]
            ax.plot(xpos, y, ls=ls, marker=mk, color=color, lw=1.6,
                    markersize=4.5, label=lab if (i == 0 and j == len(NL)-1) else None)
        ax.set_ylim(0.3, 1.03)
        ax.grid(alpha=0.25, lw=0.5)
        if i == 0:
            ax.set_title(NL_LABEL[nlk], fontsize=9)
        if j == 0:
            ax.set_ylabel(ROW_LABEL[st] + "\nfidelity", fontsize=8)
        if i == len(STRUCT)-1:
            ax.set_xticks(xpos); ax.set_xticklabels(EPS, fontsize=7, rotation=45)
        if i == 0 and j == len(NL)-1:
            ax.legend(fontsize=6, loc='lower left', framealpha=0.9)
fig.supxlabel('perturbation amplitude', fontsize=9)
fig.supylabel('Spearman fidelity to ground truth', fontsize=9)
fig.tight_layout()
OUT = os.path.join(str(OUT_DIR), "phase_diagram.png")
fig.savefig(OUT, dpi=200, bbox_inches='tight')
print("wrote", OUT, "(persistence collapsed; 3 lines per panel, matches paper)")
plt.show()

wrote /content/Emergent_Contribution/results/phase_diagram.png (persistence collapsed; 3 lines per panel, matches paper)


## 6. Deep-domain seed ensemble (V-Dem) — per-node leverage stability

Retrains the V-Dem NAVAR across 20 seeds and recomputes the per-node leverage ordering each time, reporting across-seed rank stability, non-reducibility to influence/variance baselines, and the variance/leverage dissociation. This is the worked example showing the ordering is identified at the extremes and the high-variance / low-leverage components are robustly separated.

In [ ]:
# =============================================================================
# Deep-domain seed ensemble on the V-Dem panel (16 components).
# Retrains the V-Dem NAVAR (train_NAVAR, same hyperparameters as Section 3) under
# N_SEEDS seeds, recomputes E_j each time, and reports per-node mean/sd + rank
# range, full-ordering stability, non-reducibility vs baselines, and the
# high-influence / low-leverage dissociation with its across-seed rank range.
#
# RUNS AS-IS provided the Cell-7 environment is in scope, i.e. these already
# exist in your notebook:
#   Y_train_norm, MAXLAGS, split_spec, DEVICE, COMPONENTS,
#   make_torch_jacobian_fn, emergent_contribution_from_jacobians
# Only the TRAINING SEED varies across the ensemble.
# =============================================================================
import numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from train_NAVAR import train_NAVAR
from asof_jacobian import make_torch_jacobian_fn
from emergent_contribution import emergent_contribution_from_jacobians

N = len(COMPONENTS); HORIZON = 8; N_SEEDS = 20

# --- Cell-5 variables this script depends on (defined there; fallbacks if run standalone) ---
try: TIME_COL
except NameError: TIME_COL = "year"
try: UNIT_COL
except NameError: UNIT_COL = "country_id"
try: CUTOFF_C
except NameError: CUTOFF_C = 2000
# Resolve the raw-data path robustly (do NOT trust a DATA that may have been
# overwritten elsewhere; validate that the CSV actually exists under it).
from pathlib import Path as _Path
def _resolve_data_dir(default=None):
    if default is None:
        default = str(globals().get("DATA", "data"))
    _d = globals().get("DATA", None)
    for base in ([str(_d)] if isinstance(_d, (str, _Path)) else []) + [default]:
        if os.path.isfile(os.path.join(base, "my_data_75_years.csv")):
            return base
    raise FileNotFoundError("my_data_75_years.csv not found; set DATA to its directory.")
DATA = _resolve_data_dir()

# --- exact Cell-7 hyperparameters (do not change; only seed varies) ---
HP = dict(maxlags=MAXLAGS, hidden_nodes=32, hidden_layers=1, dropout=0.0,
          epochs=400, learning_rate=1e-3, batch_size=256, lambda1=0.15,
          val_proportion=0.10, weight_decay=1e-4, check_every=40,
          normalize=False, split_timeseries=split_spec, lstm=False)

def train_one(seed):
    seed_everything(seed)                            # full determinism per member
    out = train_NAVAR(data=Y_train_norm, **HP)      # (_, _, val_loss, model)
    val_loss, model = out[2], out[3]
    model.to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    return model, float(val_loss)

# --- score E_j per node, RESPECTING country/segment boundaries -----------------
# IMPORTANT: Y_train_norm stacks all countries; sliding a window across the whole
# matrix would let a HORIZON window straddle a country boundary (meaningless
# propagation). We use the SAME segmentation that training used (split_spec) to
# score within each segment only.
Y = np.asarray(Y_train_norm, float)
T = Y.shape[0]

def _segment_bounds(spec, total):
    """Return list of (start,end) row ranges. NAVAR's `split_timeseries` convention
       (see train_NAVAR docstring) is a SINGLE INT = the length of each equal-length
       sub-series. Handle that first, then a few list formats for safety. Falls back
       to one segment with a printed warning if unrecognized."""
    import numpy as _np
    if spec is None or spec is False or (isinstance(spec,(int,_np.integer)) and int(spec)<=0):
        print("WARNING: split_spec is falsy/0 — scoring as ONE segment (verify intended).")
        return [(0,total)]
    # PRIMARY CASE: scalar int = per-segment length (balanced panel: 75 yrs x 89 countries)
    if isinstance(spec,(int,_np.integer)):
        L=int(spec)
        if total % L != 0:
            print(f"WARNING: total rows {total} not divisible by split length {L}; "
                  f"check the panel is balanced.")
        return [(i,i+L) for i in range(0, total-L+1, L)]
    arr=_np.asarray(spec)
    if arr.ndim==2 and arr.shape[1]==2:                                  # (start,end) pairs
        return [(int(a),int(b)) for a,b in arr]
    if arr.ndim==1 and len(arr)==total and len(_np.unique(arr))<total:   # per-row IDs
        bounds=[]; s=0
        for i in range(1,total+1):
            if i==total or arr[i]!=arr[s]: bounds.append((s,i)); s=i
        return bounds
    if arr.ndim==1:                                                      # list of lengths
        b=[]; pos=0
        for L in arr: b.append((pos,pos+int(L))); pos+=int(L)
        if pos==total: return b
    print(f"WARNING: split_spec format unrecognized (shape {arr.shape}); "
          f"scoring as ONE segment — VERIFY boundaries!")
    return [(0,total)]

SEG_BOUNDS=_segment_bounds(split_spec, T)
print(f"scoring over {len(SEG_BOUNDS)} segments (e.g. first few: {SEG_BOUNDS[:3]})")

def score(model):
    jacf = make_torch_jacobian_fn(model, n_components=N, maxlags=MAXLAGS,
                                  lag_reduction="companion", device=DEVICE)
    tot = np.zeros(N); cnt = 0
    for (a,b) in SEG_BOUNDS:
        if b-a < MAXLAGS+HORIZON: continue           # too short to score
        Js = [jacf(Y[s-MAXLAGS:s].T) for s in range(a+MAXLAGS, b)]  # within segment
        for t in range(len(Js)-HORIZON+2):
            ec = emergent_contribution_from_jacobians([Js[t+h] for h in range(HORIZON-1)],
                                                      n_components=N, horizon=HORIZON, strict=True)
            tot += np.array(ec.weights); cnt += 1
    w = tot/max(cnt,1); return w/w.sum()

# --- SEED ENSEMBLE ---
W=[]; vlosses=[]
for seed in range(N_SEEDS):
    m, vl = train_one(seed); W.append(score(m)); vlosses.append(vl)
    print(f"seed {seed} done  (val_loss={vl:.5f})")
W=np.array(W)
mean_w=W.mean(0); sd_w=W.std(0)
ranks=np.array([pd.Series(-w).rank().values for w in W])
rank_lo=ranks.min(0).astype(int); rank_hi=ranks.max(0).astype(int)

print(f"\nmean val_loss over seeds: {np.mean(vlosses):.5f} +/- {np.std(vlosses):.5f}")
print("\n=== PER-NODE E_j: seed mean, sd, rank range ===")
for i in np.argsort(-mean_w):
    print(f"  {COMPONENTS[i]:>24s}: w={mean_w[i]:.4f} +/- {sd_w[i]:.4f}  rank {rank_lo[i]}-{rank_hi[i]}")
ps=[spearmanr(W[a],W[b]).correlation for a in range(N_SEEDS) for b in range(a+1,N_SEEDS)]
print(f"\nmean pairwise Spearman across seeds: {np.mean(ps):.3f}")

# --- non-reducibility vs baselines + dissociation (seed-0 model) ---
m0,_=train_one(0)
jac1=make_torch_jacobian_fn(m0,n_components=N,maxlags=MAXLAGS,lag_reduction="lag1",device=DEVICE)
A_sum=np.zeros((N,N)); nw=0
for (a,b) in SEG_BOUNDS:
    if b-a < MAXLAGS+1: continue
    for s in range(a+MAXLAGS, b): A_sum+=np.abs(jac1(Y[s-MAXLAGS:s].T)); nw+=1
A_inf=A_sum/max(nw,1)
# A_inf[i,j] = |d x_i / d x_j| = influence of node j ON node i.
# Node j's OUTGOING influence (how much it drives others) = column j = sum over i = axis 0.
# Node j's INCOMING influence (how much it is driven)      = row j    = sum over j = axis 1.
out_strength = A_inf.sum(0)          # outgoing: how much each node drives the system
in_strength  = A_inf.sum(1)          # incoming: how much each node is driven
two_step     = np.abs(A_inf@A_inf).sum(0)
# --- VARIANCE on RAW (pre-standardization) values, TRAINING window, within-country ---
# BUG FIX: Y_train_norm is standardized -> every column variance is 1.0, so variance
# computed on Y is meaningless. The substantive question ("does this component MOVE a
# lot over time?") requires the RAW values, restricted to the pre-cutoff training
# window, measured as temporal variance WITHIN each country and averaged across them.
_raw = pd.read_csv(f"{DATA}/my_data_75_years.csv").sort_values([UNIT_COL, TIME_COL])
_raw_train = _raw[_raw[TIME_COL] < CUTOFF_C]              # pre-2000, matches Y_train
var_comp = _raw_train.groupby(UNIT_COL)[COMPONENTS].var().mean(0).values  # avg within-country temporal var
Ej0=score(m0)

print("\n=== non-reducibility (Spearman E_j vs baseline) ===")
for nm,b in [("out-strength",out_strength),("in-strength",in_strength),
             ("two-step-reach",two_step),("within-var",var_comp)]:
    print(f"  {nm:>16s}: {spearmanr(Ej0,b).correlation:+.3f}")

# FULL rank table — no threshold filter (the old top-6/bottom-2 filter could
# silently miss a dissociation). Read dissociations directly off this.
ej_rank   = pd.Series(-Ej0).rank().astype(int)
out_rank  = pd.Series(-out_strength).rank().astype(int)
two_rank  = pd.Series(-two_step).rank().astype(int)
var_rank  = pd.Series(-var_comp).rank().astype(int)
print("\n=== per-node: E_j rank | out-strength rank | two-step rank | variance rank | seed range ===")
for i in np.argsort(-Ej0):
    print(f"  {COMPONENTS[i]:>24s}:  Ej {int(ej_rank[i]):2d} | out {int(out_rank[i]):2d} | "
          f"2-step {int(two_rank[i]):2d} | var {int(var_rank[i]):2d} | seeds {rank_lo[i]}-{rank_hi[i]}")
print("\nDissociation = large gap between E_j rank and an influence/variance rank for the same node.")


# -----------------------------------------------------------------------------
# EMIT canonical deepdomain_ensemble.json for the reproducibility verifier.
# Keys match verifier_core.py exactly. ranks[component] carries the seed-0
# ('ej') and within-country-variance ('var') ranks used by the dissociation
# checks; n_country_years is the training-window size (rows of Y_train_norm).
# -----------------------------------------------------------------------------
import os as _os, json as _json
_dd = {
    "pairwise_spearman":  round(float(np.mean(ps)), 4),
    "corr_within_var":    round(float(spearmanr(Ej0, var_comp).correlation), 4),
    "corr_out_strength":  round(float(spearmanr(Ej0, out_strength).correlation), 4),
    "corr_two_step":      round(float(spearmanr(Ej0, two_step).correlation), 4),
    "corr_in_strength":   round(float(spearmanr(Ej0, in_strength).correlation), 4),
    "val_loss_mean":      round(float(np.mean(vlosses)), 5),
    "val_loss_sd":        round(float(np.std(vlosses)), 5),
    "n_seeds":            int(N_SEEDS),
    "n_country_years":    int(T),          # rows of Y_train_norm (training window)
    "n_segments":         int(len(SEG_BOUNDS)),
    "ranks": {
        COMPONENTS[i]: {
            "ej":       int(ej_rank[i]),
            "var":      int(var_rank[i]),
            "out":      int(out_rank[i]),
            "two_step": int(two_rank[i]),
            "seed_lo":  int(rank_lo[i]),
            "seed_hi":  int(rank_hi[i]),
            "mean_w":   round(float(mean_w[i]), 4),
            "sd_w":     round(float(sd_w[i]), 4),
        }
        for i in range(N)
    },
}
_dd_path = _os.path.join(str(OUT_DIR), "deepdomain_ensemble.json")
with open(_dd_path, "w") as _f:
    _json.dump(_dd, _f, indent=2)
print(f"\nwrote {_dd_path}")
print(f"  pairwise_spearman={_dd['pairwise_spearman']}  "
      f"within_var={_dd['corr_within_var']}  val_loss={_dd['val_loss_mean']}  "
      f"n_country_years={_dd['n_country_years']}")


scoring over 89 segments (e.g. first few: [(0, 46), (46, 92), (92, 138)])
iteration 40. Loss: 0.2860269844532013  Val loss: 0.04892151430249214
iteration 80. Loss: 0.1570504903793335  Val loss: 0.04492296278476715
iteration 120. Loss: 0.15205171704292297  Val loss: 0.04397263377904892
iteration 160. Loss: 0.15061120688915253  Val loss: 0.043242134153842926
iteration 200. Loss: 0.14980758726596832  Val loss: 0.04334787279367447
iteration 240. Loss: 0.14935103058815002  Val loss: 0.0434236042201519
iteration 280. Loss: 0.1488593965768814  Val loss: 0.04306007921695709
iteration 320. Loss: 0.14866577088832855  Val loss: 0.04337208345532417
iteration 360. Loss: 0.14854353666305542  Val loss: 0.043147072196006775
iteration 400. Loss: 0.14827819168567657  Val loss: 0.04316488653421402
seed 0 done  (val_loss=0.04316)
iteration 40. Loss: 0.28334271907806396  Val loss: 0.05002802237868309
iteration 80. Loss: 0.15765146911144257  Val loss: 0.045358169823884964
iteration 120. Loss: 0.152364745736

## 7. Verify paper numbers against the most recent run

Checks paper numbers against the most recent results.

In [ ]:
!cd {CODE} && python verify_paper_numbers.py --root {RESULTS}/ --verbose

  [PASS] T1.signflip.med    exp 0.47 (tol 0.01) got=0.47083333333333344 — Table1 sign-reversal med Delta_pooled = 0.47
  [PASS] T1.signflip.high   exp 0.58 (tol 0.01) got=0.5753968253968256 — Table1 sign-reversal high Delta_pooled = 0.58
  [PASS] T1.signflip.low    exp 0.26 (tol 0.01) got=0.25853174603174606 — Table1 sign-reversal low Delta_pooled = 0.26
  [PASS] T1.reroute.high    exp 0.1 (tol 0.01) got=0.09484126984126977 — Table1 persistent-reroute high Delta_pooled = 0.10
  [PASS] T1.static.med      exp 0.05 (tol 0.01) got=0.047023809523809496 — Table1 static med Delta_pooled = 0.05
  [PASS] T1.drift.med       exp -0.01 (tol 0.01) got=-0.0059523809523809 — Table1 smooth-drift med Delta_pooled approx -0.01
  [PASS] CI.med.lo          exp 0.32 (tol 0.01) got=0.3222123015873016 — sign-flip med CI lower = 0.32
  [PASS] CI.med.hi          exp 0.63 (tol 0.01) got=0.6307738095238096 — sign-flip med CI upper = 0.63
  [PASS] CI.high.lo         exp 0.4 (tol 0.01) got=0.39521825396825405 — si